In [1]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Importing the trial dataset for the first subject

In [2]:
subject_1_train_raw = mne.io.read_raw_gdf("../dataset/BCICIV_2a_gdf/A01T.gdf", verbose=False, preload=True)

c:\Users\akila\AppData\Local\Programs\Python\Python314\Lib\contextlib.py:148: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


In [13]:
subject_1_train_raw.annotations.description

array(['32766', '276', '32766', '277', '32766', '1072', '32766', '768',
       '772', '768', '771', '768', '770', '768', '769', '768', '769',
       '768', '770', '768', '771', '768', '772', '768', '770', '768',
       '771', '768', '769', '768', '769', '768', '769', '768', '772',
       '768', '770', '768', '770', '768', '769', '768', '769', '768',
       '771', '768', '769', '768', '770', '768', '1023', '772', '768',
       '772', '768', '771', '768', '769', '768', '772', '768', '772',
       '768', '770', '768', '772', '768', '772', '768', '770', '768',
       '769', '768', '770', '768', '771', '768', '771', '768', '771',
       '768', '772', '768', '771', '768', '769', '768', '772', '768',
       '770', '768', '771', '768', '770', '768', '771', '768', '772',
       '768', '1023', '770', '768', '771', '768', '769', '32766', '768',
       '769', '768', '769', '768', '772', '768', '770', '768', '1023',
       '769', '768', '771', '768', '769', '768', '771', '768', '1023',
       '770'

In [18]:
data, times = subject_1_train_raw[:]

In [26]:
print(subject_1_train_raw.ch_names)
print(subject_1_train_raw.get_channel_types())

['EEG-Fz', 'EEG-0', 'EEG-1', 'EEG-2', 'EEG-3', 'EEG-4', 'EEG-5', 'EEG-C3', 'EEG-6', 'EEG-Cz', 'EEG-7', 'EEG-C4', 'EEG-8', 'EEG-9', 'EEG-10', 'EEG-11', 'EEG-12', 'EEG-13', 'EEG-14', 'EEG-Pz', 'EEG-15', 'EEG-16', 'EOG-left', 'EOG-central', 'EOG-right']
['eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg', 'eeg']


In [28]:
eog_chs = subject_1_train_raw.ch_names[-3:] 
subject_1_train_raw.set_channel_types({ch: 'eog' for ch in eog_chs})

<RawGDF | A01T.gdf, 25 x 672528 (2690.1 s), ~128.3 MiB, data loaded>

In [31]:
ch_names_2a = ['Fz','FC3','FC1','FCz','FC2','FC4','C5','C3','C1','Cz', 'C2','C4','C6','CP3','CP1','CPz','CP2','CP4','P1','Pz','P2','POz','EOG-left','EOG-central','EOG-right']
subject_1_train_raw.rename_channels(dict(zip(subject_1_train_raw.ch_names, ch_names_2a)))

<RawGDF | A01T.gdf, 25 x 672528 (2690.1 s), ~128.3 MiB, data loaded>

In [33]:
print(set(subject_1_train_raw.annotations.description))

{np.str_('276'), np.str_('1072'), np.str_('1023'), np.str_('768'), np.str_('771'), np.str_('772'), np.str_('277'), np.str_('770'), np.str_('32766'), np.str_('769')}


In [57]:
marker_label = {
    1: "Rejected Trial", 
    2: "Eye movments",
    3: "Idling EEG (eyes open)",
    4: "Idling EEG (eyes closed)", \
    5: "Start of a new run",
    6: "Start of a trial",
    7: "left_hand", 
    8: "right_hand",
    9: "feet", 
    10: "tongue"
}

In [58]:
subject_1_train_raw.annotations.description = np.array(
    [marker_label.get(d, d) for d in subject_1_train_raw.annotations.description]
)

In [92]:
subject_1_train_raw.filter(l_freq=8, h_freq=30)

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 8 - 30 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 8.00
- Lower transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 7.00 Hz)
- Upper passband edge: 30.00 Hz
- Upper transition bandwidth: 7.50 Hz (-6 dB cutoff frequency: 33.75 Hz)
- Filter length: 413 samples (1.652 s)



<RawGDF | A01T.gdf, 25 x 672528 (2690.1 s), ~128.3 MiB, data loaded>

In [93]:
events, event_id = mne.events_from_annotations(subject_1_train_raw, verbose=False)
print(event_id)
print(events)

{np.str_('1023'): 1, np.str_('1072'): 2, np.str_('276'): 3, np.str_('277'): 4, np.str_('32766'): 5, np.str_('768'): 6, np.str_('769'): 7, np.str_('770'): 8, np.str_('771'): 9, np.str_('772'): 10}
[[     0      0      5]
 [     0      0      3]
 [ 29683      0      5]
 ...
 [670550      0      6]
 [670550      0      1]
 [671050      0      7]]


In [94]:
mi_event_id = {marker_label[v]: v for k, v in event_id.items() if marker_label[v] in ['left_hand', 'right_hand', 'feet', 'tongue']}
print(mi_event_id)

{'left_hand': 7, 'right_hand': 8, 'feet': 9, 'tongue': 10}


In [124]:
epochs = mne.Epochs(subject_1_train_raw, events, event_id=mi_event_id, tmin=-0.5, tmax=5.25, baseline=None, preload=True)

X = epochs.get_data()        
y = epochs.events[:, -1]     

Not setting metadata
288 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 288 events and 1438 original time points ...
0 bad epochs dropped


In [125]:
print(epochs.event_id)
print(epochs)

{'left_hand': 7, 'right_hand': 8, 'feet': 9, 'tongue': 10}
<Epochs | 288 events (all good), -0.5 – 5.248 s (baseline off), ~79.0 MiB, data loaded,
 'left_hand': 72
 'right_hand': 72
 'feet': 72
 'tongue': 72>


In [126]:
print(epochs['left_hand'].get_data().shape)
print(epochs['right_hand'].get_data().shape)
print(epochs['feet'].get_data().shape)
print(epochs['tongue'].get_data().shape)

(72, 25, 1438)
(72, 25, 1438)
(72, 25, 1438)
(72, 25, 1438)


In [127]:
epochs_mi = epochs.copy().crop(tmin=0.5, tmax=4)
epochs_mi.pick_types(eeg=True)
X = epochs_mi.get_data()   # shape (n_trials, n_eeg_channels, n_times)
y = epochs_mi.events[:, -1]

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).


In [ ]:
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

# select only left_hand and right_hand trials
binary_mask = np.isin(y, [mi_event_id['left_hand'], mi_event_id['right_hand']])
X_bin = X[binary_mask]
y_bin = y[binary_mask]

csp = CSP(n_components=8, reg=None, log=True, norm_trace=False)
lda = LinearDiscriminantAnalysis()
clf = Pipeline([('CSP', csp), ('LDA', lda)])

scores = cross_val_score(clf, X_bin, y_bin, cv=10)
print(scores, scores.mean())

Computing rank from data with rank=None
    Using tolerance 2.7e-05 (2.2e-16 eps * 22 dim * 5.4e+09  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=7 covariance using EMPIRICAL
Done.
Estimating class=8 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 2.7e-05 (2.2e-16 eps * 22 dim * 5.4e+09  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=7 covariance using EMPIRICAL
Done.
Estimating class=8 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 2.6e-05 (2.2e-16 eps * 22 dim * 5.4e+09  max singular value)
    Estimated rank (data): 22
    data: rank 22 computed from 22 data channels with 0 projectors
Reducing data rank from 22 -> 22
Estimating class=7 covariance using EMP